# Delete a training set from OMERO

Remove one training set completely: its tracking table, the ROIs and label files it uploaded (including uploads a crash left untracked, found by their tag), and its config. Everything is found through the table and its tag, never by name, so other training sets are left alone.

Deleting on OMERO cannot be undone. The delete runs as a **dry run** first and only shows what would be removed. Objects owned by another user (possible in read-write groups) block the delete unless you pass `force_others=True`.

The set's local folder (under `~/omero_annotate_ai/`) is not touched; delete it yourself if you no longer need it.

In [ ]:
from omero_annotate_ai import create_omero_connection_widget
from omero_annotate_ai.omero.omero_functions import delete_training_set
from omero_annotate_ai.trainingset import list_training_sets, summarize

## 1. Connect to OMERO

In [ ]:
conn_widget = create_omero_connection_widget()
conn_widget.display()

In [ ]:
conn = conn_widget.get_connection()
if conn is None:
    raise ConnectionError(
        "Not connected yet: use the widget above, then rerun this cell."
    )

## 2. Find the training set

List the training sets on a dataset, project, plate or screen.

In [ ]:
container_type = "dataset"  # dataset, project, plate or screen
container_id = 0  # replace with your container ID

for info in list_training_sets(conn, container_type, container_id):
    summary = summarize(info.table_id, conn)
    done = sum(d for d, _ in summary.progress.values())
    total = sum(t for _, t in summary.progress.values())
    print(f"Table {info.table_id}: {info.name} ({done}/{total} annotated)")

## 3. Dry run

Shows what would be deleted. Nothing is removed yet.

In [ ]:
table_id = 0  # replace with the table ID to delete

report = delete_training_set(conn, table_id, dry_run=True)

## 4. Delete

Check the report above, then run this cell. If it lists objects under `others`, they belong to
another user: only add `force_others=True` if you are sure they should go too.

In [ ]:
report = delete_training_set(conn, table_id, dry_run=False)

In [ ]:
conn.close()
print("OMERO connection closed.")